# Variant E — Train + Export ReLU-KAN LUT

Single-shot training run for variant E (ReLU-KAN, pure basis) that produces
`quantised.bin` containing both the feature transformer and the two i8 ReLU-KAN
LUTs (kan1, kan2). The LUT export is wired in `examples/kan_variant_e.rs` via
`relu_kan_lut_save_format()` — see commit `fc5dfdc`.

This artifact unblocks variant E for Phase D SPRT in Akimbo (kan-relu-lut branch).

**Plan**: 1 variant × 1 seed × ~10 min on T4, then copy the final checkpoint
directory to Drive.

**Runtime**: GPU (T4 or better).


## 1. Install Rust + clone repo

In [ ]:
%%bash
if ! command -v cargo &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    echo 'source $HOME/.cargo/env' >> ~/.bashrc
fi
source $HOME/.cargo/env
rustc --version
cargo --version

In [ ]:
%%bash
set -e
if [ -d /content/bullet ]; then
    cd /content/bullet
    git fetch origin
    git reset --hard origin/main
else
    cd /content
    git clone https://github.com/y0sif/bullet.git
    cd bullet
fi
git log -1 --oneline

## 2. Download training data (test77 binpack)

In [ ]:
%%bash
apt-get install -y zstd 2>/dev/null || true

mkdir -p /content/bullet/data
cd /content/bullet/data

if [ ! -f test77.binpack ]; then
    echo "Downloading test77 binpack from HuggingFace (~1.3 GB compressed)..."
    wget -q -O test77.binpack.zst \
        "https://huggingface.co/datasets/linrock/test77/resolve/main/test77-2022-01-jan-2tb7p.binpack.zst"
    echo "Download complete. Decompressing..."
    zstd -d test77.binpack.zst -o test77.binpack --rm
    echo "Done!"
fi

ls -lh test77.binpack

## 3. Build variant E

In [ ]:
%%bash
source $HOME/.cargo/env
cd /content/bullet
cargo build --release --example kan_variant_e 2>&1 | tail -3

## 4. Run variant E (1 seed, 40 superbatches)

Streams the trainer log to `/content/variant_e_export_log.txt`. Final checkpoint
will land in `/content/bullet/checkpoints/kan-variant-e/40/`, which contains
`quantised.bin` (feature transformer + kan1 LUT + kan2 LUT).

In [ ]:
import subprocess, sys, os, shutil

os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

log_path = "/content/variant_e_export_log.txt"
ckpt_dir = "/content/bullet/checkpoints"

# Clean any leftover checkpoints so the new run is the only thing in there
if os.path.isdir(ckpt_dir):
    shutil.rmtree(ckpt_dir, ignore_errors=True)

print(f"{'='*70}\n training kan_variant_e  ->  {log_path}\n{'='*70}")
proc = subprocess.Popen(
    ["cargo", "run", "--release", "--example", "kan_variant_e"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd="/content/bullet", text=True, bufsize=1,
)
with open(log_path, "w") as log:
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        log.write(line)
proc.wait()
print(f"\nExit code: {proc.returncode}")
if proc.returncode != 0:
    raise SystemExit("Training failed — inspect the log above.")

## 5. Verify the LUT-bearing checkpoint exists and inspect sizes

In [ ]:
%%bash
echo "Checkpoint tree:"
find /content/bullet/checkpoints -maxdepth 3 -type d | sort
echo
echo "Files in the final superbatch directory:"
ls -lh /content/bullet/checkpoints/kan-variant-e/40/ 2>/dev/null || \
  ls -lh /content/bullet/checkpoints/kan-variant-e/*/ | tail -20

echo
echo "quantised.bin size (should be FT + kan1 LUT + kan2 LUT):"
find /content/bullet/checkpoints -name 'quantised.bin' -exec ls -lh {} \;

## 6. Copy the final checkpoint to Drive

Saves the entire final superbatch directory (including `quantised.bin` for Akimbo
plus the raw float weights for any future re-export) and the training log.

In [ ]:
import shutil, os, glob
from google.colab import drive
drive.mount('/content/drive')

dest = '/content/drive/MyDrive/kanue/variant_e_export'
os.makedirs(dest, exist_ok=True)

# Copy the entire kan-variant-e checkpoints tree
src_root = '/content/bullet/checkpoints/kan-variant-e'
if os.path.isdir(src_root):
    dest_ckpt = os.path.join(dest, 'kan-variant-e')
    if os.path.isdir(dest_ckpt):
        shutil.rmtree(dest_ckpt)
    shutil.copytree(src_root, dest_ckpt)
    print(f"Copied checkpoints -> {dest_ckpt}")
else:
    print(f"WARNING: no checkpoint tree at {src_root}")

# Copy the training log
log_src = '/content/variant_e_export_log.txt'
if os.path.exists(log_src):
    shutil.copy(log_src, dest)
    print(f"Copied log -> {dest}")

print()
print("Files in Drive destination:")
for f in sorted(glob.glob(f'{dest}/**/*', recursive=True)):
    if os.path.isfile(f):
        print(f"  {os.path.getsize(f):>12,}  {f}")